In [1]:
def unary_encode(n: int) -> str:
    return "1" * (n - 1) + "0"

def unary_decode(code: str) -> int:
    count = 0
    for bit in code:
        if bit == "1":
            count += 1
        else:
            break
    return count + 1

nums = [1, 2, 3, 5, 7, 10]

for n in nums:
    encoded = unary_encode(n)
    decoded = unary_decode(encoded)
    print(f"n={n}  →  encoded={encoded}  →  decoded={decoded}")


n=1  →  encoded=0  →  decoded=1
n=2  →  encoded=10  →  decoded=2
n=3  →  encoded=110  →  decoded=3
n=5  →  encoded=11110  →  decoded=5
n=7  →  encoded=1111110  →  decoded=7
n=10  →  encoded=1111111110  →  decoded=10


In [5]:
def gamma_encode(n: int) -> str:
    binary = bin(n)[2:]
    L = len(binary)
    prefix = unary_encode(L)
    suffix = binary[1:]
    return prefix + suffix

def gamma_decode(code: str) -> int:
    L = unary_decode(code)
    prefix_len = L
    suffix = code[prefix_len : prefix_len + (L - 1)]
    binary = "1" + suffix
    return int(binary, 2)

nums = [1, 2, 3, 4, 5, 7, 10, 15, 31]

for n in nums:
    encoded = gamma_encode(n)
    decoded = gamma_decode(encoded)
    print(f"n = {n:<2}  →  encoded = {encoded:<15}  →  decoded = {decoded}")



n = 1   →  encoded = 0                →  decoded = 1
n = 2   →  encoded = 100              →  decoded = 2
n = 3   →  encoded = 101              →  decoded = 3
n = 4   →  encoded = 11000            →  decoded = 4
n = 5   →  encoded = 11001            →  decoded = 5
n = 7   →  encoded = 11011            →  decoded = 7
n = 10  →  encoded = 1110010          →  decoded = 10
n = 15  →  encoded = 1110111          →  decoded = 15
n = 31  →  encoded = 111101111        →  decoded = 31


In [6]:
def fib_sequence_upto(n: int):
    fibs = [1, 2]
    while fibs[-1] < n:
        fibs.append(fibs[-1] + fibs[-2])
    return fibs

def fib_encode(n: int) -> str:
    fibs = fib_sequence_upto(n)
    code = []
    i = len(fibs) - 1

    while i >= 0:
        if fibs[i] <= n:
            code.append("1")
            n -= fibs[i]
        else:
            code.append("0")
        i -= 1

    while code and code[0] == "0":
        code.pop(0)

    return "".join(code) + "1"

def fib_decode(code: str) -> int:
    code = code[:-1]  # remove terminator
    fibs = [1, 2]
    while len(fibs) < len(code):
        fibs.append(fibs[-1] + fibs[-2])

    n = 0
    for bit, f in zip(code[::-1], fibs):
        if bit == "1":
            n += f
    return n

nums = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30]

for n in nums:
    encoded = fib_encode(n)
    decoded = fib_decode(encoded)
    print(f"n = {n:<2}  →  encoded = {encoded:<12}  →  decoded = {decoded}")



n = 1   →  encoded = 11            →  decoded = 1
n = 2   →  encoded = 101           →  decoded = 2
n = 3   →  encoded = 1001          →  decoded = 3
n = 4   →  encoded = 1011          →  decoded = 4
n = 5   →  encoded = 10001         →  decoded = 5
n = 7   →  encoded = 10101         →  decoded = 7
n = 10  →  encoded = 100101        →  decoded = 10
n = 15  →  encoded = 1000101       →  decoded = 15
n = 20  →  encoded = 1010101       →  decoded = 20
n = 30  →  encoded = 10100011      →  decoded = 30


In [ ]:
import random
import string
from collections import defaultdict


def random_word():
    length = random.randint(3, 10)
    return ''.join(random.choice(string.ascii_lowercase) for _ in range(length))

vocab = set()
while len(vocab) < 1000:
    vocab.add(random_word())

vocab = list(vocab)


pairs = [
    (random.choice(vocab), random.randint(1, 10000))
    for _ in range(1_000_000)
]



inverted = defaultdict(list)

for word, doc in pairs:
    inverted[word].append(doc)


for word in inverted:
    inverted[word] = sorted(set(inverted[word]))


def gap_encode(lst):
    if not lst:
        return []
    gaps = [lst[0]]
    for i in range(1, len(lst)):
        gaps.append(lst[i] - lst[i-1])
    return gaps

gap_inverted = {w: gap_encode(lst) for w, lst in inverted.items()}


def encode_list(lst, encoder):
    return ''.join(encoder(x) for x in lst)

encoded_unary = {w: encode_list(gaps, unary_encode) for w, gaps in gap_inverted.items()}
encoded_gamma = {w: encode_list(gaps, gamma_encode) for w, gaps in gap_inverted.items()}
encoded_fib   = {w: encode_list(gaps, fib_encode)   for w, gaps in gap_inverted.items()}

print("Data simulation + encoding completed successfully!")
print("Example word:", vocab[0])
print("Raw docIDs:", inverted[vocab[0]][:10])
print("Gap encoded:", gap_inverted[vocab[0]][:10])
print("Unary encoded (first 50 bits):", encoded_unary[vocab[0]][:50])
print("Gamma encoded (first 50 bits):", encoded_gamma[vocab[0]][:50])
print("Fibonacci encoded (first 50 bits):", encoded_fib[vocab[0]][:50])


Data simulation + encoding completed successfully!
Example word: dhbc
Raw docIDs: [9, 17, 34, 36, 45, 56, 62, 71, 77, 95]
Gap encoded: [9, 8, 17, 2, 9, 11, 6, 9, 6, 18]
Unary encoded (first 50 bits): 11111111011111110111111111111111101011111111011111
Gamma encoded (first 50 bits): 11100011110000111100001100111000111100111101011100
Fibonacci encoded (first 50 bits): 10001110000110010111011000111010011001110001110011


In [9]:
import time
import numpy as np


def size_bits_uncompressed(lst):
    return sum(len(bin(x)[2:]) for x in lst)

sizes_raw = []
sizes_unary = []
sizes_gamma = []
sizes_fib = []

for w in inverted:
    raw_bits = size_bits_uncompressed(inverted[w])
    unary_bits = len(encoded_unary[w])
    gamma_bits = len(encoded_gamma[w])
    fib_bits = len(encoded_fib[w])

    sizes_raw.append(raw_bits)
    sizes_unary.append(unary_bits)
    sizes_gamma.append(gamma_bits)
    sizes_fib.append(fib_bits)

print("Average size per posting list (in bits):")
print("Raw:      ", np.mean(sizes_raw))
print("Unary:    ", np.mean(sizes_unary))
print("Gamma:    ", np.mean(sizes_gamma))
print("Fibonacci:", np.mean(sizes_fib))


# Helper: decode unary/gamma/fib list
def decode_unary_list(code):
    values = []
    i = 0
    while i < len(code):
        # unary ends at first zero
        j = i
        while code[j] == "1":
            j += 1
        j += 1
        n = unary_decode(code[i:j])
        values.append(n)
        i = j
    return values

def decode_gamma_list(code):
    values = []
    i = 0
    while i < len(code):
        L = unary_decode(code[i:])
        prefix_len = L
        total_len = prefix_len + (L - 1)
        n = gamma_decode(code[i:i+total_len])
        values.append(n)
        i += total_len
    return values

def decode_fib_list(code):
    values = []
    i = 0
    n = len(code)

    while i < n:
        j = i

        # ensure j+1 is valid
        while j + 1 < n and not (code[j] == "1" and code[j+1] == "1"):
            j += 1

        # if we reached end without finding "11", break
        if j + 1 >= n:
            break

        # extract codeword including "11"
        fib_code = code[i:j+2]
        values.append(fib_decode(fib_code))

        i = j + 2  # move past the terminator

    return values



# Pick a random word and a random docID to search for
word = random.choice(list(inverted.keys()))
target = random.choice(inverted[word])

print("\nTesting search speed for word:", word)
print("Target docID:", target)

# RAW SEARCH
start = time.time()
found_raw = target in inverted[word]
t_raw = time.time() - start

# UNARY SEARCH
start = time.time()
decoded_u = decode_unary_list(encoded_unary[word])
found_u = target in np.cumsum(decoded_u)  # convert gaps back to docIDs
t_unary = time.time() - start

# GAMMA SEARCH
start = time.time()
decoded_g = decode_gamma_list(encoded_gamma[word])
found_g = target in np.cumsum(decoded_g)
t_gamma = time.time() - start

# FIB SEARCH
start = time.time()
decoded_f = decode_fib_list(encoded_fib[word])
found_f = target in np.cumsum(decoded_f)
t_fib = time.time() - start

print("\nSearch times (seconds):")
print("Raw:      ", t_raw)
print("Unary:    ", t_unary)
print("Gamma:    ", t_gamma)
print("Fibonacci:", t_fib)


Average size per posting list (in bits):
Raw:       11768.879
Unary:     9990.829
Gamma:     5540.902
Fibonacci: 5107.268

Testing search speed for word: vwskjbmbor
Target docID: 6905

Search times (seconds):
Raw:       0.000156402587890625
Unary:     0.0012531280517578125
Gamma:     0.0186920166015625
Fibonacci: 0.0032396316528320312
